# 06 · RQ6 echo chambers

**Question.** What share of each platform's trends appears on no other platform? It says how much each platform has its own conversation, which platforms share stories, and how rare a breakout (3 or more platforms) is, which sets the balance of the forecasts' labels.

**Data.** The stored lists (`trend_items`), before topic snapshots exist. YouTube is left out (its policy forbids derived metrics), and Reddit isn't collected yet, so "nowhere else" means none of the other lists here.

**Method.** A *trend* is one platform's distinct item (url for Bluesky and Hacker News, title elsewhere). Its title goes through the pipeline's own filters and embedding model (`scripts/embed-titles.ts`), so "the same story" means what it means on the site. Two trends on different platforms match when their similarity is at least the threshold and their sightings are at most 24 hours apart. This compares trend with trend, where the pipeline compares items with topic centroids, so it's an upper bound on the pipeline's merges. The headline threshold is the pipeline's (`MATCH_THRESHOLD` in config/ranking.ts: 0.86 with nomic-embed-text-v1.5); the tables bracket it.

**Needs** `npm ci` at the repo root. The model is read from `.cache/transformers` (downloaded once, 23 MB, if it's missing). Vectors go to `research/data/`, which git ignores.

Run top to bottom. Clear the outputs before committing (research/README.md).

In [ ]:
import pandas as pd

from topnews import leadlag, db, echo, rhythms

pd.set_option("display.max_colwidth", 70)
engine = db.engine()  # read-only; RESEARCH_DATABASE_URL from .env.local
items = leadlag.one_clock(db.trend_items(engine, days=28))  # Google's US feed only  # YouTube left out
items.groupby("source_id").agg(first=("fetched_at", "min"), last=("fetched_at", "max"), fetches=("run_id", "nunique"))

## A · The phase-1 lists, whole days

Bluesky, Google Trends, Hacker News, Mastodon and Twitch have been collected since phase 1 (2026-09-30), so they give the longest common stretch. `share` counts each trend once; `share_by_fetches` weights it by how many hourly fetches it was on the list. `reach` counts platforms including the trend's own; `overlap` reads row against column: the share of the row platform's trends that the column platform also had.

In [ ]:
PHASE_1 = ["bluesky", "google_trends", "hacker_news", "mastodon", "twitch"]
items_a = items[items.source_id.isin(PHASE_1)]

In [ ]:
cp_a, dropped_a = echo.analyze(items_a, name="rq6_a")
print(f"{len(cp_a)} trends kept, {len(dropped_a)} dropped by the filters")
echo.echo_shares(cp_a).pivot(index="source_id", columns="threshold", values="share").round(2)

In [ ]:
T = echo.HEADLINE_THRESHOLD
at_t = echo.echo_shares(cp_a, thresholds=(T,)).set_index("source_id")
display(at_t[["trends", "alone", "share", "share_by_fetches"]].round(2))
display(echo.reach(cp_a, T).round(2))
echo.overlap_matrix(cp_a, T).round(2)

## B · Every list, from when all of them are collected

X, TikTok, Instagram and Pinterest started later, so this cut starts when the last of them has a list. TikTok (7 days), Instagram and Pinterest (30 days) cover longer windows than one fetch shows, so they can look lonelier than they are while the fast lists have only a short stretch to match against.

In [ ]:
items_b = echo.since_all_present(items)
items_b.fetched_at.min(), items_b.fetched_at.max()

In [ ]:
cp_b, dropped_b = echo.analyze(items_b, name="rq6_b")
print(f"{len(cp_b)} trends kept, {len(dropped_b)} dropped by the filters")
echo.echo_shares(cp_b).pivot(index="source_id", columns="threshold", values="share").round(2)

In [ ]:
T = echo.HEADLINE_THRESHOLD
at_t = echo.echo_shares(cp_b, thresholds=(T,)).set_index("source_id")
display(at_t[["trends", "alone", "share", "share_by_fetches"]].round(2))
display(echo.reach(cp_b, T).round(2))
echo.overlap_matrix(cp_b, T).round(2)

## Checking matches by eye

Each trend's best match on another platform, in similarity bands around the threshold. Read these before trusting it: the last wrong matches sit just above it (short names, such as "Luis Rojas" and "luis garcia"), and the first right ones it misses just below.

In [ ]:
T = echo.HEADLINE_THRESHOLD
for low, high in [(0.95, 1.01), (T + 0.04, 0.95), (T, T + 0.04), (T - 0.04, T), (T - 0.08, T - 0.04)]:
    print(f"[{low:.2f}, {high:.2f})")
    display(echo.near(cp_b, low, high, limit=15))

## Hashtags written as one word

Since task 2.13 the pipeline splits `#nationalcoffeeday` into words as it does `#NationalCoffeeDay`. The share of each list's titles that are one lowercase run of 12 or more letters shows where that matters most.

In [ ]:
one_word = cp_b.title.str.fullmatch(r"#?[a-z0-9]{12,}")
cp_b.assign(one_word=one_word).groupby("source_id").one_word.mean().round(2).sort_values(ascending=False)